**1. Setup**


In [1]:
# installing the dependencies
%pip install numpy pandas scikit-learn scipy -q


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import pandas as pd
from scipy.stats import loguniform

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    RandomizedSearchCV
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

**2. Load Data & clean**

In [3]:
df = pd.read_csv("diabetes.csv")

In [4]:
df.shape

(768, 9)

**3. Quick checks**

In [5]:
# check missing values
df.isnull().sum()

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64

**4. Split features & target**

In [6]:
X = df.drop(columns=["Outcome"])
y = df["Outcome"]

In [7]:
y[:5]

0    1
1    0
2    1
3    0
4    1
Name: Outcome, dtype: int64

**Model Training, Tuning, and Evaluation Flow**

Full data<br>
↓<br>
Train–test split<br>
↓<br>
GridSearchCV / RandomizedSearchCV<br>
↓<br>
CV Fold 1: fit scaler + model on train fold, validate<br>
↓<br>
CV Fold 2: fit scaler + model on train fold, validate<br>
↓<br>
CV Fold 3: ...<br>
↓<br>
Best hyperparameters selected<br>
↓<br>
Final evaluation on test set (once)

---

**Key Clarifications**

❌ Cross-validation is NOT done on the full dataset<br>
❌ Test data is NOT used during hyperparameter tuning<br>
<br>
✅ Cross-validation happens only inside the training set<br>
✅ Test set is used only once, at the very end

**5. Train Test Split**

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [9]:
print("Dataset size:", X.shape)
print("Training data size", X_train.shape)
print("Test data size:", X_test.shape)

Dataset size: (768, 8)
Training data size (614, 8)
Test data size: (154, 8)


**6. Pipeline (scaler + model)**

This creates a scikit-learn pipeline that chains two steps together:

1. StandardScaler() standardizes each feature to have roughly zero mean and unit variance.
2. SVC() trains an SVM classifier on the scaled data.

So when you fit the pipeline, the data is first scaled and then passed to the SVM. When you predict, the same scaling is applied automatically before the model makes a prediction.

In [10]:
pipeline = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("model", SVC())
    ]
)

**7. GridSearchCV (tries all combinations)**

This defines a hyperparameter search space for tuning an SVM model (probably via GridSearchCV or RandomizedSearchCV) that's wrapped in a scikit-learn Pipeline with a step named "model".

This defines the hyperparameter search space for a scikit-learn model, specifically the SVC inside your pipeline.

It tells GridSearchCV to try different combinations of:

1. kernel: which SVM kernel to use
2. C: regularization strength
- What C actually means (regularization strength)
C controls the trade-off between two competing goals for the SVM:

- Maximizing the margin (the gap between the decision boundary and the closest points), and
- Minimizing classification errors on the training data.

- Small C (e.g., 0.1): The model prioritizes a wide margin, even if that means misclassifying some training points. This gives a smoother, simpler decision boundary — more regularization, less risk of overfitting, but possibly underfitting.
- Large C (e.g., 100): The model tries hard to classify every training point correctly, even if that means a narrower margin and a more complex, wiggly boundary. Less regularization — higher risk of overfitting, but can capture more nuanced patterns.

3. gamma: kernel coefficient for rbf and poly
 - controls how far the influence of a single training point reaches.

- Analogy: imagine each training point as a light source. gamma is how far its light spreads. Low gamma = soft floodlight illuminating a wide area (smooth boundary). High gamma = a narrow, sharp spotlight (boundary hugs individual points tightly).

4. degree: polynomial degree for the poly kernel

What each block means:
The reason it is written as a list of dictionaries is that each kernel has different valid hyperparameters. GridSearchCV will test every allowed combination in these dictionaries and pick the best one based on cross-validation score.



In [11]:
param_grid = [
    {
        "model__kernel": ["linear"],
        "model__C": [0.1, 1, 10, 100]
    },
    {
        "model__kernel": ["rbf"],
        "model__C": [0.1, 1, 10, 100],
        "model__gamma": ["scale", "auto", 0.01, 0.1,]
    },
    {
        "model__kernel": ["poly"],
        "model__C": [0.1, 1, 10],
        "model__gamma": ["scale", "auto", 0.01, 0.1],
        "model__degree": [2, 3, 4]
    }
]

In [12]:
param_grid

[{'model__kernel': ['linear'], 'model__C': [0.1, 1, 10, 100]},
 {'model__kernel': ['rbf'],
  'model__C': [0.1, 1, 10, 100],
  'model__gamma': ['scale', 'auto', 0.01, 0.1]},
 {'model__kernel': ['poly'],
  'model__C': [0.1, 1, 10],
  'model__gamma': ['scale', 'auto', 0.01, 0.1],
  'model__degree': [2, 3, 4]}]

In [13]:
grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    return_train_score=True,
    n_jobs=-1
)

In [14]:
grid.fit(X_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...del', SVC())])"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","[{'model__C': [0.1, 1, ...], 'model__kernel': ['linear']}, {'model__C': [0.1, 1, ...], 'model__gamma': ['scale', 'auto', ...], 'model__kernel': ['rbf']}, ...]"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"return_train_score return_train_score: bool, default=FalseIf ``False``, the ``cv_results_`` attribute will not include trainingscores.Computing training scores is used to get insights on how differentparameter settings impact the overfitting/underfitting trade-off.However computing the scores on the training set can be computationallyexpensive and is not strictly required to select the parameters thatyield the best generalization performance... versionadded:: 0.19.. versionchanged:: 0.21 Default value was changed from ``True`` to ``False``",True
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params

In [15]:
print("GRID SEARCH CV - Results:")
print("Best params:", grid.best_params_)
print("Best CV accuracy:", grid.best_score_)

GRID SEARCH CV - Results:
Best params: {'model__C': 10, 'model__gamma': 0.01, 'model__kernel': 'rbf'}
Best CV accuracy: 0.7785419165667067


In [16]:
# evaluation on global test set (unseen data)
test_accuracy = round(grid.score(X_test, y_test)*100, 2)
print(f"Test Accuracy: {test_accuracy} %")

Test Accuracy: 75.32 %


**Analysing results**

In [17]:
grid.cv_results_

{'mean_fit_time': array([0.01934314, 0.01818819, 0.01772704, 0.15597897, 0.00615458,
        0.00467281, 0.007902  , 0.00681305, 0.01025252, 0.00756803,
        0.02292066, 0.01225619, 0.01396098, 0.01139169, 0.00916758,
        0.009097  , 0.01369162, 0.02065487, 0.00844598, 0.01378384,
        0.00598226, 0.00594177, 0.00582657, 0.00592046, 0.00591049,
        0.00464826, 0.00578642, 0.00817118, 0.00635948, 0.00560799,
        0.00580988, 0.00761867, 0.00674214, 0.00601969, 0.006953  ,
        0.00463099, 0.00448098, 0.00565863, 0.00424719, 0.00590687,
        0.00833707, 0.0063024 , 0.00541658, 0.00702481, 0.01381712,
        0.01048422, 0.00382214, 0.01029873, 0.00893564, 0.00958767,
        0.00606384, 0.00912352, 0.00982094, 0.00800476, 0.00624123,
        0.00747085]),
 'std_fit_time': array([0.00324845, 0.00360688, 0.00466234, 0.07705674, 0.00242049,
        0.0015292 , 0.00308772, 0.00211448, 0.00641134, 0.00409264,
        0.02862628, 0.00556857, 0.00775862, 0.00334729, 0.003

In [18]:
# gris search cv - all cv results

grid_results_df = pd.DataFrame(grid.cv_results_)

# select only the useful columns
grid_results_df = grid_results_df[["params", "mean_train_score", "mean_test_score", "rank_test_score"]].sort_values("rank_test_score")

grid_results_df

,params,mean_train_score,mean_test_score,rank_test_score
14,"{'model__C': 10, 'model__gamma': 0.01, 'model_...",0.800496,0.778542,1
0,"{'model__C': 0.1, 'model__kernel': 'linear'}",0.790718,0.778529,2
10,"{'model__C': 1, 'model__gamma': 0.01, 'model__...",0.789904,0.776929,3
2,"{'model__C': 10, 'model__kernel': 'linear'}",0.794791,0.775290,4
1,"{'model__C': 1, 'model__kernel': 'linear'}",0.795197,0.775290,4
3,"{'model__C': 100, 'model__kernel': 'linear'}",0.795198,0.773664,6
18,"{'model__C': 100, 'model__gamma': 0.01, 'model...",0.827775,0.762308,7
11,"{'model__C': 1, 'model__gamma': 0.1, 'model__k...",0.833473,0.758963,8
4,"{'model__C': 0.1, 'model__gamma': 'scale', 'mo...",0.779728,0.754085,9
5,"{'model__C': 0.1, 'model__gamma': 'auto', 'mod...",0.779728,0.754085,9


In [19]:
len(grid_results_df)

56

**8. Randomized Search CV (random combinations)**

In [20]:
param_dist = [
    # Linear
    {
        "model__kernel": ["linear"],
        "model__C": loguniform(0.01, 100)
    },
 
    # RBF
    {
        "model__kernel": ["rbf"],
        "model__C": loguniform(0.01, 1000),
        "model__gamma": loguniform(0.0001, 10)
    },

    # Poly
    {
        "model__kernel": ["poly"],
        "model__C": loguniform(0.01, 100),
        "model__gamma": loguniform(0.0001, 1),
    }
]

In [21]:
random_search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_dist,
    n_iter=20,
    cv=5,
    scoring="accuracy",
    random_state=42,
    return_train_score=True,
    n_jobs=-1
)

In [23]:
random_search.fit(X_train, y_train)

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...del', SVC())])"
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","[{'model__C': <scipy.stats....t 0x115bd16a0>, 'model__kernel': ['linear']}, {'model__C': <scipy.stats....t 0x115ba5bd0>, 'model__gamma': <scipy.stats....t 0x115ba56d0>, 'model__kernel': ['rbf']}, ...]"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",20
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"return_train_score return_train_score: bool, default=FalseIf ``False``, the ``cv_results_`` attribute will not include trainingscores.Computing training scores is used to get insights on how differentparameter settings impact the overfitting/underfitting trade-off.However computing the scores on the training set can be computationallyexpensive and is not strictly required to select the parameters thatyield the best generalization performance... versionadded:: 0.19.. versionchanged:: 0.21 Default value was changed from ``True`` to ``False``",True
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `st

In [24]:
print("Randomized SEARCH CV - Results:")
print("Best params:", random_search.best_params_)
print("Best CV accuracy:", random_search.best_score_)

Randomized SEARCH CV - Results:
Best params: {'model__C': np.float64(4.205156450913866), 'model__gamma': np.float64(0.01444525102276306), 'model__kernel': 'rbf'}
Best CV accuracy: 0.7850593096094894


In [25]:
# evaluation on global test set (unseen dataset)
test_accuracy = round(random_search.score(X_test, y_test)*100, 2)
print(f"Test accuracy: {test_accuracy} %")

Test accuracy: 74.03 %


In [26]:
# -----------------------------
# RandomizedSearchCV - All CV results
# -----------------------------
random_search_results_df = pd.DataFrame(random_search.cv_results_)

# Select only the most useful columns
random_search_results_df = random_search_results_df[
    [
        "params",
        "mean_train_score",
        "mean_test_score",
        "rank_test_score"
    ]
].sort_values("rank_test_score")

random_search_results_df

,params,mean_train_score,mean_test_score,rank_test_score
7,"{'model__C': 4.205156450913866, 'model__gamma'...",0.799275,0.785059,1
3,"{'model__C': 2.5378155082656657, 'model__kerne...",0.794383,0.775290,2
8,"{'model__C': 1.256315277393867, 'model__kernel...",0.793569,0.775290,2
12,"{'model__C': 0.015339162591163621, 'model__ker...",0.782573,0.773677,4
1,"{'model__C': 2.440060709081755, 'model__kernel...",0.793976,0.773664,5
6,"{'model__C': 2.950706670790534, 'model__kernel...",0.794383,0.773664,5
16,"{'model__C': 26.373339933815235, 'model__gamma...",0.824923,0.763934,7
17,"{'model__C': 2.754143921332031, 'model__gamma'...",0.868490,0.737892,8
10,"{'model__C': 0.6672367170464207, 'model__gamma...",0.809042,0.731361,9
4,"{'model__C': 0.012087541473056965, 'model__gam...",0.835916,0.729708,10


In [27]:
random_search_results_df[:1]["params"].values

array([{'model__C': np.float64(4.205156450913866), 'model__gamma': np.float64(0.01444525102276306), 'model__kernel': 'rbf'}],
      dtype=object)